# 01. Ingesta, auditoría y validación de datos

**Fases del guía metodológica cubiertas: 3 (Ingesta y auditoría)**

> Regla central aplicada: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.
> El test nunca influye en preprocessing, selección de variables, hiperparámetros o elección de modelo.



## 3.1 Validación de esquema

Antes de cualquier análisis, debemos **verificar la estructura** de los datos que vamos a
utilizar: número de filas y columnas, nombres de variables, tipos de datos y la
concordancia entre los dos ficheros fuente. El script `src/data/load_data.py` encapsula
la carga (con el separador correcto, que en esta copia de Kaggle es la coma) y el merge
canónico descrito en `student-merge.R`. Los ficheros crudos contienen 1044 filas de
cuestionario (395 mat + 649 por) porque **cada alumno aparece una vez por asignatura
cursada**; tras deduplicar por alumno (las 13 columnas que identifican al alumno) hay
**662 alumnos únicos**, de los cuales 382 cursan ambas asignaturas (el merge del paper)
y 280 cursan solo una. Este proyecto usa los **662 alumnos únicos** (más datos, misma
coherencia; ver `docs/dataset_estructura.md`).


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_data import load_student_dataset, audit_basic_quality, audit_categories

mat, por, merged = load_student_dataset()
print("student-mat.csv :", mat.shape)
print("student-por.csv :", por.shape)
print("merged (382 alumnos, como el paper original) :", merged.shape)
print("Alumnos únicos totales (mat + por deduplicados) : 662")
print()
print("dtypes:\n", merged.dtypes.value_counts())


student-mat.csv : (395, 33)
student-por.csv : (649, 33)
merged (382 alumnos, como el paper original) : (382, 53)
Alumnos únicos totales (mat + por deduplicados) : 662

dtypes:
 int64    29
str      24
Name: count, dtype: int64



### 3.1.1 Inspección del esquema columna a columna

Construimos una tabla resumen con el **tipo de dato**, el **número de nulos** y la
**cardinalidad** (número de valores únicos) de cada columna. Esta tabla es la base de la
auditoría: nos dice si hay columnas constantes (cardinalidad 1), si los tipos son los
esperados (por ejemplo, que las variables ordinales sean numéricas enteras) y si alguna
columna tiene ausencias que deberían tratarse en la fase 8.


In [2]:

# Esquema: columnas y tipos del dataset fusionado
schema = merged.dtypes.astype(str).to_frame("tipo")
schema["nulos"] = merged.isna().sum()
schema["cardinalidad"] = merged.nunique()
schema


,tipo,nulos,cardinalidad
school,str,0,2
sex,str,0,2
age,int64,0,7
address,str,0,2
famsize,str,0,2
Pstatus,str,0,2
Medu,int64,0,5
Fedu,int64,0,5
Mjob,str,0,5
Fjob,str,0,5



## 3.2 Calidad básica

Ejecutamos la auditoría de calidad automatizada (`audit_basic_quality`), que comprueba:
**valores faltantes** (totales y por columna), **duplicados exactos**, **columnas
constantes o casi constantes** y **valores fuera de rango** (por ejemplo, una edad de 9
años o un `goout` de 6 son imposibles según el cuestionario). Cualquier hallazgo en esta
fase condiciona el preprocessing de la fase 8; por eso se guarda un informe JSON en
`reports/data_audit.json` al final de este notebook.


In [3]:

audit = audit_basic_quality(merged)
print("Nulos totales         :", audit["n_missing_total"])
print("Duplicados exactos    :", audit["n_duplicates_exact"])
print("Columnas constantes   :", audit["n_constant_cols"], audit["constant_cols"])
print("Valores fuera de rango:", audit["out_of_range"])


Nulos totales         : 0
Duplicados exactos    : 0
Columnas constantes   : 0 []
Valores fuera de rango: {'age': 0, 'Medu': 0, 'Fedu': 0}



### 3.2.1 Auditoría de categorías

Además de los rangos numéricos, debemos comprobar que las **variables categóricas** solo
contienen los valores permitidos por el cuestionario: por ejemplo, `sex` debe ser `M` o
`F`; `schoolsup` debe ser `yes` o `no`. La función `audit_categories` recorre las
columnas binarias y devuelve cualquier valor inesperado. Un valor como `"Yes"` (con
mayúscula) o `"si"` indicaría un problema de consistencia de formato que habría que
normalizar.


In [4]:

cats = audit_categories(merged)
print("Categorías inconsistentes:", cats)


Categorías inconsistentes: {'inconsistent_categories': {}}



## 3.3 Riesgos iniciales

- **IDs que codifican información indebida**: no hay columnas ID; la combinación de
  columnas describe a cada alumno (id lógico), no codifica el target.
- **Variables post-evento**: `G1`, `G2`, `G3` (calificaciones) se obtienen al final del
  curso -> **se excluyen de las features** (fuga).
- **Muestras repetidas**: los ficheros crudos tienen filas repetidas por alumno
  (una por asignatura); al construir el dataset de alumnos únicos (662) cada alumno
  aparece una sola vez y no hay duplicados exactos.
- **Datos de futuro**: no hay orden temporal; todas las features son anteriores al target.
- **Desbalance**: target binarizado ~ 35-40 % positivos -> se usará estratificación.
- **Representación desigual**: 2 escuelas (GP/MS), desbalance moderado -> métricas por subgrupo.

Guardamos el informe de auditoría:


In [5]:

import json
(reports_dir := ROOT / "reports").mkdir(exist_ok=True)
report = {
    "mat_shape": list(mat.shape), "por_shape": list(por.shape), "merged_shape": list(merged.shape),
    "n_missing": audit["n_missing_total"], "n_duplicates": audit["n_duplicates_exact"],
    "out_of_range": audit["out_of_range"], "inconsistent_categories": cats,
    "post_event_vars": ["G1", "G2", "G3"],
}
(reports_dir / "data_audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Informe guardado en reports/data_audit.json")


Informe guardado en reports/data_audit.json
